In [1]:
import pandas as pd

def load_clean():
    df = pd.read_csv('dabba_dash_orders.csv')
    df = df.drop_duplicates()
    df['city'] = df['city'].str.strip().replace({
        'mumbai': 'Mumbai', 'New Delhi': 'Delhi',
        'BENGALURU': 'Bengaluru', 'Bangalore': 'Bengaluru', 'bengaluru': 'Bengaluru'})
    df['delivery_time_min'] = df['delivery_time_min'].where(df['delivery_time_min'] <= 180)
    return df

df = load_clean()
df.shape, df['delivery_time_min'].mean()

((15000, 14), np.float64(25.655287201643176))

In [2]:
def city_report(df, city):
    sub = df[df['city'] == city]                      # all orders in this city
    delivered = sub[sub['status'] == 'Delivered']     # only delivered orders
    return {
        'city': city,
        'orders': len(sub),
        'revenue': delivered['order_value_inr'].sum(),
        'cancel_rate_pct': round((sub['status'] == 'Cancelled').mean() * 100, 1),
        'avg_delivery_min': round(sub['delivery_time_min'].mean(), 1),
    }

city_report(df, 'Mumbai')

{'city': 'Mumbai',
 'orders': 4278,
 'revenue': np.float64(1853938.0),
 'cancel_rate_pct': np.float64(5.5),
 'avg_delivery_min': np.float64(26.7)}

In [3]:
cities = list(df['city'].unique())

rows = []
for city in cities:
    rows.append(city_report(df, city))

report = pd.DataFrame(rows).sort_values('revenue', ascending=False)
report.to_csv('city_report.csv', index=False)
report

,city,orders,revenue,cancel_rate_pct,avg_delivery_min
2,Mumbai,4278,1853938.0,5.5,26.7
0,Bengaluru,3600,1401347.0,5.8,27.6
4,Delhi,2643,1081290.0,5.1,25.6
3,Pune,1767,654349.0,6.1,22.5
5,Hyderabad,1651,605039.0,5.9,24.5
1,Thane,1061,373753.0,4.4,22.2


In [4]:
df.groupby('city')['distance_km'].mean().round(2).sort_values(ascending=False)

city
Bengaluru    3.11
Hyderabad    3.10
Mumbai       3.10
Delhi        3.07
Pune         3.03
Thane        2.99
Name: distance_km, dtype: float64

In [5]:
import os

os.makedirs('monthly_files', exist_ok=True)

df['month'] = pd.to_datetime(df['order_date'], format='%d-%m-%Y').dt.strftime('%Y-%m')

for month in sorted(df['month'].unique()):
    df[df['month'] == month].to_csv(f'monthly_files/orders_{month}.csv', index=False)

len(os.listdir('monthly_files'))

18

In [6]:
len(pd.read_csv('monthly_files/orders_2025-10.csv'))

1163

In [7]:
import glob

files = sorted(glob.glob('monthly_files/orders_*.csv'))

parts = []
for f in files:
    parts.append(pd.read_csv(f))

combined = pd.concat(parts, ignore_index=True)
combined.shape

(15000, 15)

In [8]:
combined[combined['status'] == 'Delivered']['order_value_inr'].sum()

np.float64(5969716.0)

In [9]:
combined.shape
combined[combined['status'] == 'Delivered']['order_value_inr'].sum()

np.float64(5969716.0)

In [10]:
print(files[:3])
print(len(files))

['monthly_files\\orders_2025-01.csv', 'monthly_files\\orders_2025-02.csv', 'monthly_files\\orders_2025-03.csv']
18


In [11]:
import requests

url = 'https://api.open-meteo.com/v1/forecast'
params = {
    'latitude': 19.07,
    'longitude': 72.88,
    'daily': 'temperature_2m_max,precipitation_sum',
    'timezone': 'Asia/Kolkata',
}

response = requests.get(url, params=params)
print(response.status_code)

data = response.json()
print(data.keys())

200
dict_keys(['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'daily_units', 'daily'])


In [12]:
print(data['daily_units'])
print(data['daily'])

{'time': 'iso8601', 'temperature_2m_max': '°C', 'precipitation_sum': 'mm'}
{'time': ['2026-11-08', '2026-11-09', '2026-11-10', '2026-11-11', '2026-11-12', '2026-11-13', '2026-11-14'], 'temperature_2m_max': [36.6, 35.6, 35.7, 35.1, 34.3, 34.5, 34.2], 'precipitation_sum': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]}


In [13]:
weather = pd.DataFrame(data['daily'])
weather.to_csv('mumbai_weather.csv', index=False)
weather

,time,temperature_2m_max,precipitation_sum
0,2026-11-08,36.6,0.0
1,2026-11-09,35.6,0.0
2,2026-11-10,35.7,0.0
3,2026-11-11,35.1,0.0
4,2026-11-12,34.3,0.0
5,2026-11-13,34.5,0.0
6,2026-11-14,34.2,0.0


In [14]:
coords = {
    'Mumbai': (19.07, 72.88), 'Bengaluru': (12.97, 77.59), 'Delhi': (28.61, 77.21),
    'Pune': (18.52, 73.86), 'Hyderabad': (17.39, 78.49), 'Thane': (19.22, 72.98),
}

def get_weather(city, lat, lon):
    params = {'latitude': lat, 'longitude': lon,
              'daily': 'temperature_2m_max,precipitation_sum', 'timezone': 'Asia/Kolkata'}
    response = requests.get('https://api.open-meteo.com/v1/forecast', params=params)
    response.raise_for_status()
    table = pd.DataFrame(response.json()['daily'])
    table['city'] = city
    return table

weather_parts = []
for city, (lat, lon) in coords.items():
    weather_parts.append(get_weather(city, lat, lon))

all_weather = pd.concat(weather_parts, ignore_index=True)
all_weather.to_csv('city_weather.csv', index=False)
all_weather.shape

(42, 4)